In [ ]:
!pip install kaggle==1.5.16

In [ ]:
! chmod 600 .kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

In [ ]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip

# TimesFM — Walmart Store Sales Forecasting (Bonus)

**Model:** Google's TimesFM — a pretrained time-series foundation model (decoder-only
Transformer, patch-based, trained on 100M+ real-world series).

**Two modes tested:**
1. **Zero-shot** — no training at all, straight inference using Google's pretrained weights
2. **Fine-tuned** — the pretrained model is further trained on Walmart's own sales history

**Note:** TimesFM's API has changed across versions and is under active development.
If a cell errors due to an API mismatch, check the official quickstart at
https://github.com/google-research/timesfm for the current signature — the core
logic here (load → zero-shot forecast → fine-tune → forecast again) should remain valid
even if exact method names shift.

In [ ]:
!pip uninstall -y torchvision torchaudio -q
!pip install "numpy<2" "torch>=2.4" timesfm[torch] transformers accelerate peft mlflow dagshub wandb -q --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
import torch
import wandb
import mlflow
import mlflow.pyfunc

import timesfm

WANDB_ENTITY  = 'ikvas22-free-university-of-tbilisi'
WANDB_PROJECT = 'Walmart Weekly Sales Prediction'
WANDB_GROUP   = 'TimesFM'

MLFLOW_EXPERIMENT = 'TimesFM_Training'
MLFLOW_MODEL_NAME  = 'timesfm_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H          = 4     # forecast horizon (weeks) — same as other 3 models
INPUT_SIZE = 52     # context length given to TimesFM
VAL_START  = '2012-04-01'
N_WINDOWS  = 7       # same rolling windows as evaluate_cv in other notebooks

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

wandb.login()

print(f'Setup OK — device: {DEVICE}')

## 1. Pre-processing

In [ ]:
run = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'preprocessing',
    name     = 'TimesFM_Preprocessing',
)

train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

df = (
    train_raw
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
)

wandb.log({
    'raw_rows' : df.shape[0],
    'raw_cols' : df.shape[1],
    'stores'   : df['Store'].nunique(),
    'depts'    : df['Dept'].nunique(),
    'date_min' : str(df['Date'].min().date()),
    'date_max' : str(df['Date'].max().date()),
})

# TimesFM (like N-BEATS/DLinear) only needs the raw sales series —
# it's a univariate foundation model, no engineered features consumed.
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df_nf = (
    df[['unique_id', 'Date', 'Weekly_Sales', 'IsHoliday']]
    .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

min_len    = INPUT_SIZE + H
series_len = df_nf.groupby('unique_id')['ds'].count()
valid_ids  = series_len[series_len >= min_len].index
dropped    = series_len[series_len < min_len].index.tolist()
df_nf      = df_nf[df_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)

wandb.log({
    'total_series'   : len(series_len),
    'valid_series'   : len(valid_ids),
    'dropped_series' : len(dropped),
})

print(f'Valid series : {len(valid_ids)}')
print(f'Dropped      : {len(dropped)} (shorter than {min_len} weeks)')

df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()
df_full  = pd.concat([df_train, df_val]).sort_values(['unique_id', 'ds']).reset_index(drop=True)

wandb.log({
    'train_rows'     : len(df_train),
    'val_rows'       : len(df_val),
    'val_start'      : VAL_START,
    'horizon_weeks'  : H,
    'context_weeks'  : INPUT_SIZE,
})

print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')

wandb.finish()

## 2. Training

Since TimesFM is not a `neuralforecast` model, we can't use `NeuralForecast.cross_validation()`
directly. Instead we replicate the same rolling-window logic manually — same `N_WINDOWS=7`,
same `step_size=H` — so the WMAE is directly comparable to N-BEATS / DLinear / TFT.

In [3]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def rolling_windows(df_series: pd.DataFrame, n_windows: int, h: int, input_size: int):
    """
    Yields (context_dates, target_dates) pairs for each rolling window,
    mirroring neuralforecast's cross_validation(n_windows, step_size=h).
    Windows walk backward from the end of df_series in steps of h.
    """
    dates = sorted(df_series['ds'].unique())
    for w in range(n_windows, 0, -1):
        cutoff_idx = len(dates) - w * h
        if cutoff_idx < input_size:
            continue
        context_dates = dates[max(0, cutoff_idx - input_size):cutoff_idx]
        target_dates  = dates[cutoff_idx:cutoff_idx + h]
        yield context_dates, target_dates


def evaluate_timesfm(tfm_model, df_source: pd.DataFrame, n_windows: int = N_WINDOWS, batch_size: int = 64) -> tuple:
    all_preds = []
    unique_ids = df_source['unique_id'].unique()

    # Collect all (uid, context, target) tuples across all windows first
    tasks = []
    for uid in unique_ids:
        s = df_source[df_source['unique_id'] == uid].sort_values('ds')
        for context_dates, target_dates in rolling_windows(s, n_windows, H, INPUT_SIZE):
            context = s[s['ds'].isin(context_dates)]['y'].values
            target  = s[s['ds'].isin(target_dates)]
            if len(context) < 1 or len(target) == 0:
                continue
            tasks.append((uid, context, target))

    print(f'Total forecast tasks: {len(tasks):,}  (batched in groups of {batch_size})')

    # Run forecasts in batches instead of one-by-one
    for i in range(0, len(tasks), batch_size):
        batch = tasks[i:i + batch_size]
        contexts = [t[1] for t in batch]

        point_forecast, _ = tfm_model.forecast(H, contexts)

        for j, (uid, context, target) in enumerate(batch):
            pred = point_forecast[j][:len(target)]
            for k, (_, row) in enumerate(target.iterrows()):
                all_preds.append({
                    'unique_id' : uid,
                    'ds'        : row['ds'],
                    'y'         : row['y'],
                    'y_pred'    : pred[k] if k < len(pred) else np.nan,
                    'IsHoliday' : row['IsHoliday'],
                })

        if (i // batch_size) % 20 == 0:
            print(f'  {i + len(batch)}/{len(tasks)} tasks done')

    eval_df = pd.DataFrame(all_preds).dropna(subset=['y_pred'])
    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['IsHoliday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [ ]:
# ── Zero-shot: load pretrained TimesFM, no training ────────────
run_zeroshot = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'TimesFM_ZeroShot',
    config   = {
        'checkpoint'  : 'google/timesfm-2.5-200m-pytorch',
        'context_len' : INPUT_SIZE,
        'horizon'     : H,
        'mode'        : 'zero-shot',
    }
)

tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
tfm.compile(
    timesfm.ForecastConfig(
        max_context = INPUT_SIZE,
        max_horizon = H,
        normalize_inputs = True,
        use_continuous_quantile_head = True,
    )
)

zeroshot_wmae, zeroshot_mae, eval_zeroshot = evaluate_timesfm(tfm, df_full, n_windows=N_WINDOWS)

wandb.log({'val_wmae': zeroshot_wmae, 'val_mae': zeroshot_mae})
print(f'Zero-shot WMAE : {zeroshot_wmae:,.2f}')
print(f'Zero-shot MAE  : {zeroshot_mae:,.2f}')

wandb.finish()

In [ ]:
!pip install "torchao>=0.16.0" -q --force-reinstall

In [ ]:
# ── Fine-tuning: LoRA via HuggingFace Transformers + PEFT ──────
# Based on the official example:
# https://github.com/google-research/timesfm/tree/master/timesfm-forecasting/examples/finetuning
#
# NOTE: uses a DIFFERENT model class/checkpoint than zero-shot above.
# Zero-shot used `timesfm.TimesFM_2p5_200M_torch` (pip package's own wrapper,
# auto-pads any context length). Fine-tuning needs `TimesFm2_5ModelForPrediction`
# from HuggingFace Transformers, since that's what PEFT/LoRA attaches to — and
# it requires context length to be a multiple of 32 (patch size). So fine-tuning
# uses CONTEXT_LEN_FT=64 rather than INPUT_SIZE=52 used elsewhere in this
# notebook. The zero-shot/fine-tuned WMAE numbers below are therefore NOT on
# an identical context window — noted explicitly so the comparison isn't overstated.

# !pip install transformers accelerate peft -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TimesFm2_5ModelForPrediction
from peft import LoraConfig, get_peft_model

CONTEXT_LEN_FT = 64     # must be a multiple of 32 (patch length)
HORIZON_LEN_FT = H       # keep the same forecast horizon as the rest of the project
FT_EPOCHS      = 3       # small for runtime; official default is 10 — raise if time allows
FT_LR          = 1e-4
FT_BATCH_SIZE  = 32
FT_NUM_SAMPLES = 3000
LORA_R         = 4
LORA_ALPHA     = 8
LORA_DROPOUT   = 0.05


class TimeSeriesRandomWindowDataset(Dataset):
    """Random-window dataset, same approach as the official example: pre-samples
    random (series, split-point) windows so every example has a full, unpadded
    context (avoids corrupting TimesFM's internal instance normalisation)."""

    def __init__(self, series_list, context_len, horizon_len, num_samples=3000, seed=42):
        self.series_list = series_list
        self.context_len = context_len
        self.horizon_len = horizon_len
        self.samples = []

        rng = np.random.default_rng(seed)
        min_len = context_len + horizon_len
        valid = [i for i, s in enumerate(series_list) if len(s) >= min_len]
        if not valid:
            raise ValueError(f"No series long enough for context_len={context_len} + horizon_len={horizon_len}")

        for _ in range(num_samples):
            idx = rng.choice(valid)
            series = series_list[idx]
            max_start = len(series) - min_len
            start = rng.integers(0, max_start + 1)
            self.samples.append((idx, start))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        idx, start = self.samples[i]
        series = self.series_list[idx]
        end = start + self.context_len + self.horizon_len
        context = torch.tensor(series[start:start + self.context_len], dtype=torch.float32)
        target  = torch.tensor(series[start + self.context_len:end], dtype=torch.float32)
        return context, target


class TimeSeriesLastWindowDataset(Dataset):
    """Validation dataset using the last window of each series."""

    def __init__(self, series_list, context_len, horizon_len):
        self.items = []
        min_len = context_len + horizon_len
        for s in series_list:
            if len(s) >= min_len:
                ctx = torch.tensor(s[-min_len:-horizon_len], dtype=torch.float32)
                tgt = torch.tensor(s[-horizon_len:], dtype=torch.float32)
                self.items.append((ctx, tgt))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


run_finetune = wandb.init(
    entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
    job_type="training", name="TimesFM_FineTune_LoRA",
    config={
        "model_id": "google/timesfm-2.5-200m-transformers",
        "context_len_ft": CONTEXT_LEN_FT,
        "horizon_len_ft": HORIZON_LEN_FT,
        "epochs": FT_EPOCHS, "lr": FT_LR, "batch_size": FT_BATCH_SIZE,
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "num_samples": FT_NUM_SAMPLES,
    }
)

# Build per-series arrays from df_train (chronological Weekly_Sales history)
all_series = [
    g.sort_values("ds")["y"].values.astype(np.float32)
    for _, g in df_train.groupby("unique_id")
]
all_series = [s for s in all_series if len(s) >= CONTEXT_LEN_FT + HORIZON_LEN_FT]
print(f"Series usable for fine-tuning (>= {CONTEXT_LEN_FT + HORIZON_LEN_FT} weeks): {len(all_series)}")

train_ds = TimeSeriesRandomWindowDataset(all_series, CONTEXT_LEN_FT, HORIZON_LEN_FT, num_samples=FT_NUM_SAMPLES)
val_ds   = TimeSeriesLastWindowDataset(all_series, CONTEXT_LEN_FT, HORIZON_LEN_FT)
train_loader = DataLoader(train_ds, batch_size=FT_BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=FT_BATCH_SIZE)
print(f"Train windows: {len(train_ds)} ({len(train_loader)} batches)  Val windows: {len(val_ds)}")

ft_dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32
ft_model = TimesFm2_5ModelForPrediction.from_pretrained(
    "google/timesfm-2.5-200m-transformers", torch_dtype=ft_dtype, device_map=DEVICE,
)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, target_modules="all-linear",
    lora_dropout=LORA_DROPOUT, bias="none",
)
ft_model = get_peft_model(ft_model, lora_config)
ft_model.print_trainable_parameters()

optimizer = torch.optim.AdamW(ft_model.parameters(), lr=FT_LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS * len(train_loader))

best_val_loss = float("inf")
ADAPTER_DIR = "timesfm_walmart_lora"

for epoch in range(1, FT_EPOCHS + 1):
    ft_model.train()
    epoch_loss, n_batches = 0.0, 0
    for context, target_vals in train_loader:
        context, target_vals = context.to(DEVICE), target_vals.to(DEVICE)
        outputs = ft_model(past_values=context, future_values=target_vals, forecast_context_len=CONTEXT_LEN_FT)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), max_norm=1.0)
        optimizer.step(); optimizer.zero_grad(); scheduler.step()
        epoch_loss += loss.item(); n_batches += 1
    avg_train_loss = epoch_loss / max(n_batches, 1)

    ft_model.eval()
    val_loss, val_batches = 0.0, 0
    with torch.no_grad():
        for context, target_vals in val_loader:
            context, target_vals = context.to(DEVICE), target_vals.to(DEVICE)
            outputs = ft_model(past_values=context, future_values=target_vals, forecast_context_len=CONTEXT_LEN_FT)
            val_loss += outputs.loss.item(); val_batches += 1
    avg_val_loss = val_loss / max(val_batches, 1)

    print(f"Epoch {epoch}/{FT_EPOCHS} — train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}")
    wandb.log({"epoch": epoch, "train_loss": avg_train_loss, "val_loss": avg_val_loss})

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        ft_model.save_pretrained(ADAPTER_DIR)
        print(f"  saved best adapter -> {ADAPTER_DIR}")

wandb.log({"best_val_loss": best_val_loss})
print(f"Fine-tuning complete. Best val loss: {best_val_loss:.4f}")

In [ ]:
def evaluate_timesfm_hf(model, df_source, context_len=CONTEXT_LEN_FT, horizon=HORIZON_LEN_FT,
                          n_windows=N_WINDOWS, batch_size=64):
    model.eval()
    tasks = []
    for uid in df_source["unique_id"].unique():
        s = df_source[df_source["unique_id"] == uid].sort_values("ds")
        for context_dates, target_dates in rolling_windows(s, n_windows, horizon, context_len):
            context = s[s["ds"].isin(context_dates)]["y"].values
            target  = s[s["ds"].isin(target_dates)]
            if len(context) != context_len or len(target) == 0:
                continue
            tasks.append((uid, context, target))

    all_preds = []
    for i in range(0, len(tasks), batch_size):
        batch = tasks[i:i + batch_size]
        ctx_tensor = torch.tensor(np.stack([t[1] for t in batch]), dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            out = model(past_values=ctx_tensor)
        preds = out.mean_predictions.float().cpu().numpy()
        for j, (uid, context, target) in enumerate(batch):
            pred = preds[j][:len(target)]
            for k, (_, row) in enumerate(target.iterrows()):
                all_preds.append({
                    "unique_id": uid, "ds": row["ds"], "y": row["y"],
                    "y_pred": pred[k] if k < len(pred) else np.nan,
                    "IsHoliday": row["IsHoliday"],
                })

    eval_df = pd.DataFrame(all_preds).dropna(subset=["y_pred"])
    score_wmae = wmae(eval_df["y"].values, eval_df["y_pred"].values, eval_df["IsHoliday"].values)
    score_mae  = float(np.mean(np.abs(eval_df["y"].values - eval_df["y_pred"].values)))
    return score_wmae, score_mae, eval_df


finetune_wmae, finetune_mae, eval_finetune = evaluate_timesfm_hf(ft_model, df_full)

wandb.log({
    "val_wmae": finetune_wmae,
    "val_mae": finetune_mae,
    "zeroshot_wmae": zeroshot_wmae,
    "wmae_improvement": zeroshot_wmae - finetune_wmae,
})
print(f"Fine-tuned WMAE : {finetune_wmae:,.2f}  (context_len={CONTEXT_LEN_FT})")
print(f"Zero-shot WMAE  : {zeroshot_wmae:,.2f}  (context_len={INPUT_SIZE})")
print("Note: different context lengths — see markdown note above before comparing directly.")

if finetune_wmae < zeroshot_wmae:
    best_wmae, best_mae, eval_best = finetune_wmae, finetune_mae, eval_finetune
    best_mode = "fine-tuned-lora"
else:
    best_wmae, best_mae, eval_best = zeroshot_wmae, zeroshot_mae, eval_zeroshot
    best_mode = "zero-shot"

print(f"\nBest mode: {best_mode}  (WMAE {best_wmae:,.2f})")

wandb.finish()

In [ ]:
run_finetune = wandb.init(
    entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
    job_type="training", name="TimesFM_FineTune_LoRA",
    config={
        "model_id": "google/timesfm-2.5-200m-transformers",
        "context_len_ft": CONTEXT_LEN_FT,
        "horizon_len_ft": HORIZON_LEN_FT,
        "epochs": FT_EPOCHS, "lr": FT_LR, "batch_size": FT_BATCH_SIZE,
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "num_samples": FT_NUM_SAMPLES,
    }
)

wandb.log({
    "val_wmae": finetune_wmae,
    "val_mae": finetune_mae,
    "zeroshot_wmae": zeroshot_wmae,
    "wmae_improvement": zeroshot_wmae - finetune_wmae,
})
print(f"Fine-tuned WMAE : {finetune_wmae:,.2f}  (context_len={CONTEXT_LEN_FT})")
print(f"Zero-shot WMAE  : {zeroshot_wmae:,.2f}  (context_len={INPUT_SIZE})")
print("Note: different context lengths — see markdown note above before comparing directly.")

if finetune_wmae < zeroshot_wmae:
    best_wmae, best_mae, eval_best = finetune_wmae, finetune_mae, eval_finetune
    best_mode = "fine-tuned-lora"
else:
    best_wmae, best_mae, eval_best = zeroshot_wmae, zeroshot_mae, eval_zeroshot
    best_mode = "zero-shot"

print(f"\nBest mode: {best_mode}  (WMAE {best_wmae:,.2f})")

wandb.finish()

## 3. Save Best Model to MLflow Registry

In [ ]:
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

In [ ]:
class TimesFMWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import torch
        from transformers import TimesFm2_5ModelForPrediction
        from peft import PeftModel

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.context_len = 64   # CONTEXT_LEN_FT — must match training
        self.horizon     = 4    # H

        base = TimesFm2_5ModelForPrediction.from_pretrained(
            "google/timesfm-2.5-200m-transformers",
            torch_dtype=torch.float32,
            device_map=self.device,
        )
        self.model = PeftModel.from_pretrained(base, context.artifacts["adapter_dir"])
        self.model.eval()

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw DataFrame with [Store, Dept, Date, Weekly_Sales].
        Returns [Store, Dept, Date, Weekly_Sales_pred] for the next
        H weeks after each series' last observed date.
        """
        import torch

        df_in = model_input.copy()
        df_in["Date"]      = pd.to_datetime(df_in["Date"])
        df_in["unique_id"] = df_in["Store"].astype(str) + "_" + df_in["Dept"].astype(str)

        results = []
        for uid, group in df_in.groupby("unique_id"):
            g = group.sort_values("Date")
            context_vals = g["Weekly_Sales"].values[-self.context_len:]
            if len(context_vals) < self.context_len:
                continue  # LoRA-tuned model needs the full 64-week context

            ctx_tensor = torch.tensor(context_vals, dtype=torch.float32).unsqueeze(0).to(self.device)
            with torch.no_grad():
                out = self.model(past_values=ctx_tensor)
            pred = out.mean_predictions[0, :self.horizon].float().cpu().numpy()

            last_date = g["Date"].max()
            future_dates = pd.date_range(
                start=last_date + pd.Timedelta(weeks=1), periods=self.horizon, freq="W-FRI"
            )
            store, dept = uid.split("_")
            for d, p in zip(future_dates, pred):
                results.append({
                    "Store": int(store), "Dept": int(dept),
                    "Date": d, "Weekly_Sales_pred": float(p)
                })

        return pd.DataFrame(results)


with mlflow.start_run(run_name="TimesFM_Best_Model") as run:
    mlflow.log_params({
        "mode":          best_mode,
        "model_id":      "google/timesfm-2.5-200m-transformers",
        "context_len":   CONTEXT_LEN_FT,
        "horizon":       H,
        "lora_r":        LORA_R,
        "lora_alpha":    LORA_ALPHA,
        "lora_dropout":  LORA_DROPOUT,
    })
    mlflow.log_metrics({
        "val_wmae":       best_wmae,
        "val_mae":        best_mae,
        "zeroshot_wmae":  zeroshot_wmae,
        "best_val_loss":  best_val_loss,
    })

    mlflow.pyfunc.log_model(
        artifact_path         = "timesfm_model",
        python_model          = TimesFMWrapper(),
        artifacts             = {"adapter_dir": ADAPTER_DIR},
        registered_model_name = MLFLOW_MODEL_NAME,
        pip_requirements      = ["torch>=2.4", "transformers", "peft", "pandas", "numpy<2"],
    )

    print(f"Registered: {MLFLOW_MODEL_NAME}")
    print(f"Run ID    : {run.info.run_id}")
    print(f"Best mode : {best_mode}")
    print(f"Best WMAE : {best_wmae:,.2f}")

In [ ]:
loaded = mlflow.pyfunc.load_model(f"models:/{MLFLOW_MODEL_NAME}/latest")
sample = train_raw[train_raw["Store"] == 1][["Store", "Dept", "Date", "Weekly_Sales"]].head(100)
result = loaded.predict(sample)
print("Registry load OK")
print(result.head())